In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Gold Layer: Enrich, Aggregate & Score (v6 — Full Rewrite)
# MAGIC
# MAGIC **Input** : `virtue_foundation.ghana.silver_facilities_cleaned`  (90 cols, ~978 rows)
# MAGIC **Outputs**: `virtue_foundation.ghana.gold_facilities_enriched`   (113 cols)
# MAGIC             `virtue_foundation.ghana.gold_regional_summary`      (37 cols)
# MAGIC
# MAGIC ## Processing pipeline (in order):
# MAGIC  1. Read silver & second-pass region consolidation
# MAGIC  2. Second-pass clinical array cleaning (belt-and-suspenders junk filter)
# MAGIC  3. Recompute array counts after second-pass cleaning
# MAGIC  4. phone_numbers → ARRAY<STRING> conversion (gold schema requires array)
# MAGIC  5. Specialty presence flags (taxonomy-first + free-text search)
# MAGIC  6. Operator / classification booleans
# MAGIC  7. NGO Ghana filter
# MAGIC  8. Statistical anomaly flags (data-calibrated thresholds)
# MAGIC  9. citation_row_id (deterministic MD5)
# MAGIC 10. Gold citations (silver citations + step_id field)
# MAGIC 11. Preliminary facility-level medical desert score
# MAGIC 12. Write gold_facilities_enriched (exact 113-col schema)
# MAGIC 13. Build gold_regional_summary (exact 37-col schema)
# MAGIC 14. MLflow tracing

# COMMAND ----------
# MAGIC %md ## 0. Imports & Config

# COMMAND ----------

import re
import json
from datetime import datetime, timezone
from functools import reduce
from typing import Optional, List

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    ArrayType, StringType, FloatType, IntegerType, BooleanType,
    DoubleType, StructType, StructField, LongType,
)
from pyspark.sql.functions import pandas_udf

spark = SparkSession.builder.getOrCreate()
print(f"Run : {datetime.now(timezone.utc).isoformat()}")
print(f"Spark: {spark.version}")

# COMMAND ----------

class Config:
    SILVER          = "virtue_foundation.ghana.silver_facilities_cleaned"
    GOLD_FACILITIES = "virtue_foundation.ghana.gold_facilities_enriched"
    GOLD_REGIONAL   = "virtue_foundation.ghana.gold_regional_summary"
    EXTRACTION_VER  = "gold_v6"

cfg = Config()

# ── Canonical Ghana regions ────────────────────────────────────────────────
CANONICAL_REGIONS = frozenset({
    "Greater Accra", "Ashanti", "Western", "Eastern", "Central",
    "Volta", "Northern", "Upper East", "Upper West",
    "Oti", "Bono East", "Ahafo", "Savannah", "North East",
    "Western North", "Brong-Ahafo",
})

# ── Ghana population (GSS 2021) ────────────────────────────────────────────
GHANA_POPULATION = {
    "Greater Accra": 5_401_964, "Ashanti":      5_780_438,
    "Western":       2_376_021, "Central":      2_563_228,
    "Eastern":       2_633_154, "Volta":        1_693_143,
    "Northern":      2_479_461, "Brong-Ahafo":  1_724_847,
    "Upper East":    1_046_545, "Upper West":     901_683,
    "Oti":             663_338, "Bono East":    1_141_427,
    "Ahafo":           564_397, "Savannah":       606_986,
    "North East":      527_170, "Western North":  782_022,
    "Unknown":       1_000_000,
}

# COMMAND ----------
# MAGIC %md ## 1. Read Silver + Second-Pass Region Consolidation

# COMMAND ----------

silver = spark.table(cfg.SILVER)
print(f"Silver rows    : {silver.count():,}")
print(f"Silver columns : {len(silver.columns)}")

# ── Second-pass region consolidation ──────────────────────────────────────
_REGION_SECOND_PASS = {
    # Municipality / district variants still slipping through
    "Ga East Municipality":                 "Greater Accra",
    "Ga East Municipality, Greater Accra":  "Greater Accra",
    "Shai Osudoku District, Greater Accra": "Greater Accra",
    "Ledzokuku-Krowor":                     "Greater Accra",
    "Tema West Municipal":                  "Greater Accra",
    "Accra East":                           "Greater Accra",
    "Accra North":                          "Greater Accra",
    "East Legon":                           "Greater Accra",
    "North Legon":                          "Greater Accra",
    "Adenta Municipal":                     "Greater Accra",
    "Asokore Mampong Municipal":            "Ashanti",
    "Ejisu Municipal":                      "Ashanti",
    "Ejisu-Juaben Municipal":               "Ashanti",
    "Ahafo Ano South-East":                 "Ashanti",
    "Ahafo Ano South East":                 "Ashanti",
    "Kumasi Metropolitan":                  "Ashanti",
    "ASHANTI":                              "Ashanti",
    "SH":                                   "Ashanti",
    "KEEA":                                 "Central",
    "Cape Coast Municipal":                 "Central",
    "Central Ghana":                        "Central",
    "Dormaa East":                          "Brong-Ahafo",
    "Bono":                                 "Brong-Ahafo",
    "Techiman Municipal":                   "Bono East",
    "Sissala West District":                "Upper West",
    "Sissala East District":                "Upper West",
    "Central Tongu District":               "Volta",
    "South Tongu":                          "Volta",
    "Tamale Metropolitan":                  "Northern",
    "Ghana":                                "Greater Accra",
    # Self-map every canonical region
    **{r: r for r in CANONICAL_REGIONS},
}

_rmap_expr = F.create_map(*[F.lit(x) for pair in _REGION_SECOND_PASS.items() for x in pair])

silver = silver.withColumn(
    "region_normalised",
    F.when(
        F.col("region_normalised").isNull() | (F.trim(F.col("region_normalised")) == ""),
        F.lit("Unknown"),
    )
    .when(
        F.col("region_normalised").isin(list(CANONICAL_REGIONS)),
        F.col("region_normalised"),
    )
    .when(
        _rmap_expr[F.col("region_normalised")].isNotNull(),
        _rmap_expr[F.col("region_normalised")],
    )
    .otherwise(F.lit("Unknown"))
)

print("Region distribution after second-pass consolidation:")
silver.groupBy("region_normalised").count().orderBy(F.desc("count")).show(25)
unk_ct = silver.filter(F.col("region_normalised") == "Unknown").count()
total  = silver.count()
print(f"Unknown region: {unk_ct:,} / {total:,}  ({unk_ct / total * 100:.1f}%)  target < 15%")

# COMMAND ----------
# MAGIC %md ## 2. Second-Pass Clinical Array Cleaning
# MAGIC
# MAGIC Catches patterns that slip through the silver junk filter.

# COMMAND ----------

_GOLD_JUNK_RE = re.compile(
    r"("
    # Phone / contact noise (new patterns)
    r"has\s+a\s+telephone\s+number\s+at"
    r"|contact\s+phone\s+number"
    r"|has\s+(a\s+)?phone\s+number"
    r"|has\s+(an?\s+)?email\s+address"
    r"|has\s+a\s+location\s+at"
    r"|has\s+a\s+website\s+at"
    # Purely numeric strings
    r"|^\d[\d\s\-\+\(\)]*$"
    # Social media metadata
    r"|^\d+\s+(?:people\s+)?(?:like|follow|check.?in)"
    r"|\brating\s*[:\-]?\s*[\d\.]+"
    r"|^\d+\s+reviews?"
    # Admin / ownership / listing noise
    r"|\bis\s+(?:privately|government|publicly)\s+owned\b"
    r"|\bOwnership\s*[:\-]"
    r"|\bFacility\s+type\s*[:\-]"
    r"|\bType\s*[:\-]\s*(?:Primary|Secondary|Tertiary|Community|District|Regional)"
    r"|\bNHIS\s+(?:accredited|registered|accreditation)\b"
    r"|\bprovides\s+general\s+(?:medical|health)\s+services?\b"
    r"|\blisted\s+(?:as|in|on)\s+"
    r"|\bregistered\s+with\s+(?:Ghana|GHS|MOH)"
    r"|\bpage\s+(?:created|verified)"
    # Web / social
    r"|http[s]?://"
    r"|www\."
    r"|facebook\b|instagram\b|twitter\b"
    # Price range noise
    r"|price\s+range\s+\$"
    # 24/7 alone (not clinical)
    r"|^24/7$|^24\s+hours$|^always\s+open$"
    # LLM hallucination guard
    r"|we\s+should\s+not\s+make"
    r"|wait\s*[\-–]\s*we\s+should"
    r")",
    re.IGNORECASE,
)


def _gold_clean_array(arr: Optional[list]) -> list:
    """Remove second-pass junk; deduplicate by normalised key."""
    if not arr:
        return []
    seen, out = set(), []
    for v in arr:
        if not v:
            continue
        s = str(v).strip()
        # Length guard (stricter: 10 chars after stripping punctuation)
        stripped = re.sub(r"[^\w\s]", "", s)
        if len(stripped.strip()) < 10:
            continue
        if _GOLD_JUNK_RE.search(s):
            continue
        key = re.sub(r"[^\w\s]", "", s.lower())
        key = re.sub(r"\s+", " ", key).strip()
        if key not in seen:
            seen.add(key)
            out.append(s)
    return out


_gold_clean_udf = F.udf(_gold_clean_array, ArrayType(StringType()))

for _col in ("procedure_parsed", "equipment_parsed", "capability_parsed", "specialties_parsed"):
    silver = silver.withColumn(_col, _gold_clean_udf(F.col(_col)))

# Recompute counts after second-pass cleaning
silver = silver \
    .withColumn("procedure_count",  F.size(F.col("procedure_parsed"))) \
    .withColumn("equipment_count",  F.size(F.col("equipment_parsed"))) \
    .withColumn("capability_count", F.size(F.col("capability_parsed"))) \
    .withColumn("specialty_count",  F.size(F.col("specialties_parsed")))

# Recompute boolean flags
silver = silver \
    .withColumn("has_procedures",  F.col("procedure_count")  > 0) \
    .withColumn("has_equipment",   F.col("equipment_count")  > 0) \
    .withColumn("has_capabilities",F.col("capability_count") > 0) \
    .withColumn("has_specialties", F.col("specialty_count")  > 0)

print("Second-pass cleaning complete. Array count distributions:")
for _c in ("procedure_count", "equipment_count", "capability_count", "specialty_count"):
    _avg = silver.agg(F.avg(_c)).collect()[0][0] or 0
    _max = silver.agg(F.max(_c)).collect()[0][0] or 0
    _nz  = silver.filter(F.col(_c) > 0).count()
    print(f"  {_c:<20}  avg={_avg:.2f}  max={_max}  non-zero={_nz}")

# COMMAND ----------
# MAGIC %md ## 3. Convert phone_numbers → ARRAY<STRING>
# MAGIC
# MAGIC Gold schema: `phone_numbers ARRAY<STRING>`.
# MAGIC Silver schema: `phone_numbers STRING` (raw JSON) + `phone_numbers_parsed ARRAY<STRING>`.
# MAGIC Solution: overwrite `phone_numbers` column with the already-parsed `phone_numbers_parsed`.

# COMMAND ----------

# Replace STRING phone_numbers with the proper ARRAY version
gold = silver \
    .withColumn("phone_numbers",
                F.when(
                    F.col("phone_numbers_parsed").isNotNull(),
                    F.col("phone_numbers_parsed"),
                ).otherwise(F.array()))

print("phone_numbers column converted to ARRAY<STRING>.")
gold.select("name", "phone_numbers", "phone_numbers_parsed").show(5, truncate=80)

# COMMAND ----------
# MAGIC %md ## 4. Specialty Presence Flags
# MAGIC
# MAGIC Strategy (two-tier):
# MAGIC **Tier 1** — Exact taxonomy match in `specialties_parsed` (highest confidence).
# MAGIC **Tier 2** — Free-text keyword search across capability/procedure/equipment/description.
# MAGIC
# MAGIC Uses the full specialties taxonomy actually present in the data.

# COMMAND ----------

# ── Complete specialty taxonomy → flag mapping ─────────────────────────────
# Built from every unique specialty value found in the actual silver dataset.

SPEC_TAXONOMY = {
    "has_emergency_medicine": [
        "emergencyMedicine",
        "emergencyPreparednessAndDisasterResponse",
        "eyeTraumaAndEmergencyEyeCare",
        "emergencyMedicalServices",
    ],
    "has_obstetrics": [
        "gynecologyAndObstetrics",
        "obstetricsAndMaternityCare",
        "maternalFetalMedicineOrPerinatology",
        "maternalAndChildHealth",
        "reproductiveEndocrinologyAndInfertility",
        "gynecologicOncology",
        "gynecologicalOncology",
        "familyPlanningAndComplexContraception",
    ],
    "has_surgery": [
        "generalSurgery",
        "orthopedicSurgery",
        "cardiacSurgery",
        "plasticSurgery",
        "neurosurgery",
        "pediatricSurgery",
        "vascularSurgery",
        "hepatobiliarySurgery",
        "hepatopancreatobiliarySurgery",
        "colorectalSurgery",
        "transplantSurgery",
        "spineNeurosurgery",
        "maxillofacialSurgery",
        "oralAndMaxillofacialSurgery",
        "craniofacialAndCleftOralMaxillofacialSurgery",
        "dentoalveolarSurgery",
        "urologicOncology",
        "refractiveSurgeryOphthalmology",
        "cataractAndAnteriorSegmentSurgery",
    ],
    "has_pediatrics": [
        "pediatrics",
        "neonatologyPerinatalMedicine",
        "pediatricCardiology",
        "pediatricDentistry",
        "pediatricEndocrinology",
        "pediatricHematologyOncology",
        "pediatricNeurodevelopmentalDisabilities",
        "pediatricNeurosurgery",
        "pediatricSurgery",
        "pediatricUrology",
        "paediatricDentistry",
        "paediatricOphthalmology",
        "pediatricsAndStrabismusOphthalmology",
        "adolescentMedicine",
        "childAndAdolescentPsychiatry",
    ],
    "has_icu": [
        "criticalCareMedicine",
        "intensiveCare",
        "criticalCare",
        "intensiveCareUnit",
        "neonatalIntensiveCare",
    ],
    "has_radiology": [
        "radiology",
        "diagnosticRadiology",
        "interventionalRadiology",
        "abdominalRadiology",
        "breastImaging",
        "musculoskeletalRadiology",
        "nuclearMedicineAndMolecularImaging",
    ],
    "has_infectious_disease": [
        "infectiousDiseases",
        "infectiousDisease",
        "communicableDisease",
    ],
    "has_mental_health": [
        "psychiatry",
        "addictionPsychiatry",
        "childAndAdolescentPsychiatry",
        "clinicalPsychology",
        "communityAndPublicPsychiatry",
        "socialAndBehavioralSciences",
        "sleepMedicine",
    ],
}

# ── Free-text keywords for Tier 2 search ──────────────────────────────────
TEXT_KEYWORDS = {
    "has_emergency_medicine": [
        "emergency department", "emergency room", "emergency unit", "emergency ward",
        "emergency care", "accident and emergency", "a&e", "casualty department",
        "casualty ward", "trauma center", "trauma care", "trauma unit",
        "resuscitation", "urgent care", "24/7 emergency", "pre-hospital emergency",
        "emergency medical services", "ems services", "emergency response",
        "first response", "triage unit", "emergency treatment",
    ],
    "has_obstetrics": [
        "antenatal", "prenatal care", "postnatal care", "labour ward", "labor ward",
        "delivery room", "maternity ward", "maternity home", "maternity unit",
        "caesarean", "c-section", "midwifery", "obstetric care",
        "gynecology services", "gynaecology services",
        "maternal health", "safe motherhood", "family planning services",
        "reproductive health",
    ],
    "has_surgery": [
        "operating theatre", "operating room", "operation theatre",
        "surgical procedures", "laparoscopy", "laparoscopic surgery",
        "appendectomy", "hernia repair", "cholecystectomy", "thyroidectomy",
        "surgical suite", "intraoperative", "postoperative care",
        "surgical department", "general surgery services",
        "orthopaedic surgery", "orthopedic surgery", "cardiac surgery",
        "surgical unit",
    ],
    "has_pediatrics": [
        "neonatal care", "nicu", "neonatal intensive care", "child health",
        "paediatric ward", "pediatric ward", "infant care", "newborn care",
        "child and adolescent", "child welfare", "immunisation clinic",
        "vaccination clinic", "children ward", "baby care unit",
    ],
    "has_icu": [
        "intensive care unit", "intensive care", "critical care unit",
        "high dependency unit", "hdu", "high care unit",
        "mechanical ventilation", "ventilator support", "life support system",
        "neonatal intensive care", "nicu beds", "picu",
        "post-operative icu", "surgical icu",
    ],
    "has_radiology": [
        "x-ray", "x ray", "ultrasound scan", "ct scan", "mri scan",
        "mammography", "fluoroscopy", "imaging center", "diagnostic imaging",
        "medical imaging", "radiography", "sonography", "doppler ultrasound",
        "nuclear medicine", "positron emission", "pet scan",
        "radiology department", "imaging unit",
    ],
    "has_infectious_disease": [
        "hiv treatment", "aids care", "hiv/aids", "antiretroviral therapy",
        "art clinic", "tuberculosis treatment", "tb clinic", "malaria treatment",
        "pmtct", "prevention of mother to child",
        "hepatitis treatment", "viral load testing",
        "infection control", "isolation ward", "communicable disease",
        "sexually transmitted infection", "sti clinic", "cd4 testing",
        "infectious disease unit",
    ],
    "has_mental_health": [
        "mental health services", "psychiatric ward", "psychiatric hospital",
        "psychiatry unit", "psychology services", "psychotherapy",
        "counseling services", "counselling services", "mental health clinic",
        "addiction treatment", "substance abuse treatment", "drug rehabilitation",
        "alcohol rehabilitation", "depression treatment", "anxiety treatment",
        "occupational therapy for mental health", "electroconvulsive therapy",
    ],
}


@pandas_udf(BooleanType())
def make_specialty_flag_udf(
    specs:    pd.Series,  # specialties_parsed
    caps:     pd.Series,  # capability_parsed
    procs:    pd.Series,  # procedure_parsed
    equips:   pd.Series,  # equipment_parsed
    desc_col: pd.Series,  # description
    tax_json: pd.Series,  # JSON list of taxonomy terms (constant per flag)
    kw_json:  pd.Series,  # JSON list of text keywords (constant per flag)
) -> pd.Series:
    """
    Returns True if:
    - ANY taxonomy term found in specialties_parsed  (Tier 1)
    - OR ANY keyword found in combined text of cap/proc/equip/description (Tier 2)
    """
    tax = json.loads(tax_json.iloc[0])
    kws = json.loads(kw_json.iloc[0])
    tax_set = set(t.lower() for t in tax)
    kw_list = [k.lower() for k in kws]

    result = []
    for spec_v, cap_v, proc_v, equip_v, desc_v in zip(specs, caps, procs, equips, desc_col):
        # Tier 1: exact taxonomy match
        spec_list = list(spec_v) if spec_v is not None else []
        if any(s.lower() in tax_set for s in spec_list):
            result.append(True)
            continue

        # Tier 2: text search in all clinical fields
        parts = []
        for arr in [cap_v, proc_v, equip_v]:
            if arr is not None:
                parts.extend(str(x) for x in arr)
        parts.append(str(desc_v or ""))
        combined = " ".join(parts).lower()

        result.append(any(kw in combined for kw in kw_list))

    return pd.Series(result)


# Apply specialty flags
for flag_name, taxonomy in SPEC_TAXONOMY.items():
    kws  = TEXT_KEYWORDS.get(flag_name, [])
    tax_json = json.dumps(taxonomy)
    kw_json  = json.dumps(kws)

    gold = gold.withColumn(
        flag_name,
        make_specialty_flag_udf(
            F.col("specialties_parsed"),
            F.col("capability_parsed"),
            F.col("procedure_parsed"),
            F.col("equipment_parsed"),
            F.col("description"),
            F.lit(tax_json),
            F.lit(kw_json),
        )
    )

print("Specialty flags computed. Counts:")
_tot = gold.count()
for _f in ["has_emergency_medicine","has_obstetrics","has_surgery","has_pediatrics",
           "has_icu","has_radiology","has_infectious_disease","has_mental_health"]:
    _n = gold.filter(F.col(_f)).count()
    print(f"  {_f:<30}  {_n:>4} / {_tot:,}  ({_n/_tot*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 5. Operator / Classification Booleans

# COMMAND ----------

# is_public / is_private from operatorTypeId
gold = gold \
    .withColumn("is_public",
                F.when(F.lower(F.trim(F.col("operatorTypeId"))) == "public", True)
                 .otherwise(False)) \
    .withColumn("is_private",
                F.when(F.lower(F.trim(F.col("operatorTypeId"))) == "private", True)
                 .otherwise(False))

# is_hospital / is_clinic / is_ngo from facility_type_clean / organization_type_clean
gold = gold \
    .withColumn("is_hospital",
                F.when(F.lower(F.trim(F.col("facility_type_clean"))) == "hospital", True)
                 .otherwise(False)) \
    .withColumn("is_clinic",
                F.when(F.lower(F.trim(F.col("facility_type_clean"))) == "clinic", True)
                 .otherwise(False)) \
    .withColumn("is_ngo",
                F.when(
                    F.lower(F.trim(F.col("organization_type_clean"))) == "ngo",
                    True,
                ).when(
                    # Fallback: missionStatement > 20 chars and no facility type
                    F.col("missionstatement").isNotNull() &
                    (F.length(F.trim(F.col("missionstatement"))) > 20) &
                    F.col("facility_type_clean").isNull(),
                    True,
                ).otherwise(False))

print("Operator / classification flags:")
for _f in ["is_hospital", "is_clinic", "is_ngo", "is_public", "is_private"]:
    _n = gold.filter(F.col(_f)).count()
    print(f"  {_f:<18}  {_n:>4}")

# COMMAND ----------
# MAGIC %md ## 6. NGO Ghana Filter

# COMMAND ----------

gold = gold.withColumn(
    "ngo_serves_ghana",
    F.when(
        F.col("is_ngo") & (
            F.array_contains(F.col("countries_parsed"), "GH") |
            F.array_contains(F.col("countries_parsed"), "Ghana") |
            (F.lower(F.trim(F.col("address_countryCode"))) == "gh") |
            (F.lower(F.col("address_country")).contains("ghana")) |
            (
                (F.size(F.col("countries_parsed")) == 0) &   # ✅ FIXED
                (F.col("latitude").isNotNull()) &
                (F.col("latitude").between(4.0, 12.0)) &
                (F.col("longitude").between(-4.0, 2.0))
            )
        ),
        True,
    ).otherwise(False)
)

_ngo_gh = gold.filter(F.col("ngo_serves_ghana")).count()
print(f"ngo_serves_ghana: {_ngo_gh}")

# COMMAND ----------
# MAGIC %md ## 7. Statistical Anomaly Flags
# MAGIC
# MAGIC Data-calibrated thresholds based on actual silver dataset analysis:
# MAGIC - completeness scores range 0.4–1.0 (no rows below 0.25 → ghost uses 0.35 threshold)
# MAGIC - number_doctors_int: only 4 non-null rows (all hospitals) → redefine as "hospital with no doctor data"
# MAGIC - capability_count: max=19, mean=3.3 → cap>=8 is meaningful inflation threshold

# COMMAND ----------

# ── A: Capability Inflation ────────────────────────────────────────────────
# Cap claims >=8 with ZERO supporting procedures AND zero equipment (not NGO)
gold = gold.withColumn(
    "stat_anomaly_capability_inflation",
    F.when(
        (F.col("capability_count") >= 8) &
        (F.col("procedure_count") == 0) &
        (F.col("equipment_count") == 0) &
        ~F.col("is_ngo"),
        True,
    ).otherwise(False)
)

# ── B: Hospital No Doctors ─────────────────────────────────────────────────
# Is a hospital but has NO doctor data (number_doctors_int IS NULL)
# In v0.3, >99% of rows have null doctors — so this flags ALL hospitals
# without mined doctor data, which is the correct behaviour per spec.
gold = gold.withColumn(
    "stat_anomaly_hospital_no_doctors",
    F.when(
        F.col("is_hospital") &
        F.col("number_doctors_int").isNull() &
        ~F.col("is_ngo"),
        True,
    ).otherwise(False)
)

# ── C: Clinic Claims ICU ───────────────────────────────────────────────────
# Clinic-type facility has ICU evidence — clinically implausible
gold = gold.withColumn(
    "stat_anomaly_clinic_claims_icu",
    F.when(
        F.col("is_clinic") &
        F.col("has_icu") &
        ~F.col("is_ngo"),
        True,
    ).otherwise(False)
)

# ── D: Ghost Facility ──────────────────────────────────────────────────────
# Very low completeness + no contact + has capability claims (but look suspicious)
# NOTE: Silver completeness min = 0.4, so threshold is 0.5 (lower quartile)
gold = gold.withColumn(
    "stat_anomaly_ghost_facility",
    F.when(
        (F.col("data_completeness_score") < 0.50) &
        ~F.col("has_contact") &
        (F.col("capability_count") > 0) &
        (F.col("procedure_count") == 0) &
        (F.col("equipment_count") == 0) &
        ~F.col("is_ngo"),
        True,
    ).otherwise(False)
)

# ── E: Specialty-Evidence Mismatch ────────────────────────────────────────
# High specialty count (≥5) with NO procedure/equipment evidence AND thin doc
gold = gold.withColumn(
    "stat_anomaly_specialty_mismatch",
    F.when(
        (F.col("specialty_count") >= 5) &
        (F.col("procedure_count") == 0) &
        (F.col("equipment_count") == 0) &
        (F.col("doc_text_length") < 200) &
        ~F.col("is_ngo"),
        True,
    ).otherwise(False)
)

# ── F: Procedure Breadth vs Infrastructure ────────────────────────────────
# Claims many procedures (>8) but zero equipment to perform them
gold = gold.withColumn(
    "stat_anomaly_procedure_breadth",
    F.when(
        (F.col("procedure_count") > 8) &
        (F.col("equipment_count") == 0) &
        ~F.col("is_ngo"),
        True,
    ).otherwise(False)
)

# ── Total anomaly count ────────────────────────────────────────────────────
gold = gold.withColumn(
    "total_stat_anomalies",
    (
        F.col("stat_anomaly_capability_inflation") .cast(IntegerType()) +
        F.col("stat_anomaly_hospital_no_doctors")  .cast(IntegerType()) +
        F.col("stat_anomaly_clinic_claims_icu")    .cast(IntegerType()) +
        F.col("stat_anomaly_ghost_facility")       .cast(IntegerType()) +
        F.col("stat_anomaly_specialty_mismatch")   .cast(IntegerType()) +
        F.col("stat_anomaly_procedure_breadth")    .cast(IntegerType())
    )
)

print("Anomaly flags computed:")
_tot = gold.count()
for _f in ["stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
           "stat_anomaly_clinic_claims_icu", "stat_anomaly_ghost_facility",
           "stat_anomaly_specialty_mismatch", "stat_anomaly_procedure_breadth"]:
    _n = gold.filter(F.col(_f)).count()
    print(f"  {_f:<42}  {_n:>4} ({_n/_tot*100:.1f}%)")

_any_anom = gold.filter(F.col("total_stat_anomalies") > 0).count()
print(f"  Rows with at least 1 anomaly: {_any_anom:,}")

# COMMAND ----------
# MAGIC %md ## 8. citation_row_id (Deterministic MD5)

# COMMAND ----------

gold = gold.withColumn(
    "citation_row_id",
    F.md5(
        F.concat_ws(
            "|",
            F.coalesce(F.col("unique_id"),   F.lit("")),
            F.coalesce(F.col("name"),        F.lit("")),
            F.coalesce(F.col("ingested_at").cast(StringType()), F.lit("")),
        )
    )
)

# COMMAND ----------
# MAGIC %md ## 9. Gold-Level Citations (Silver citations + step_id field)
# MAGIC
# MAGIC Silver citations schema: ARRAY<STRUCT<field, snippet, source_column, extraction_method, confidence>>
# MAGIC Gold citations schema:   ARRAY<STRUCT<field, snippet, source_column, extraction_method, confidence, step_id>>

# COMMAND ----------

GOLD_CITATION_SCHEMA = ArrayType(StructType([
    StructField("field",             StringType(), True),
    StructField("snippet",           StringType(), True),
    StructField("source_column",     StringType(), True),
    StructField("extraction_method", StringType(), True),
    StructField("confidence",        FloatType(),  True),
    StructField("step_id",           StringType(), True),
]))

# Source→step_id mapping
_SRC_STEP_MAP = {
    "procedure":   "gold_v6_procedure_enrichment",
    "equipment":   "gold_v6_equipment_enrichment",
    "capability":  "gold_v6_capability_validation",
    "specialties": "gold_v6_specialty_inference",
    "description": "gold_v6_description_fallback",
}

def _upgrade_citations(silver_citations,
                       proc_arr, equip_arr, cap_arr, spec_arr,
                       has_em, has_obs, has_surg, has_ped, has_icu,
                       has_rad, has_inf, has_mh):
    """
    1. Parse silver citations and add step_id field
    2. Add flag-trigger citations for any specialty flag that fired
    """
    out = []
    seen_snippets = set()

    # ── 1. Upgrade existing silver citations ──────────────────────────────
    raw_cits = silver_citations or []
    for c in raw_cits:
        if not c:
            continue
        # In Spark, struct fields accessed as attributes
        try:
            field  = getattr(c, "field", None) or ""
            snip   = getattr(c, "snippet", None) or ""
            src    = getattr(c, "source_column", None) or ""
            method = getattr(c, "extraction_method", None) or ""
            conf   = float(getattr(c, "confidence", None) or 0.0)
        except Exception:
            continue
        step = _SRC_STEP_MAP.get(src, f"gold_v6_{src}")
        key  = (field, snip[:50])
        if key not in seen_snippets:
            seen_snippets.add(key)
            out.append({
                "field": field, "snippet": snip[:300],
                "source_column": src, "extraction_method": method,
                "confidence": conf, "step_id": step,
            })

    # ── 2. Add specialty-flag citations ───────────────────────────────────
    FLAG_SOURCES = [
        ("has_emergency_medicine", has_em, "emergencyMedicine",   "specialties"),
        ("has_obstetrics",         has_obs,"gynecologyAndObstetrics","specialties"),
        ("has_surgery",            has_surg,"generalSurgery",     "specialties"),
        ("has_pediatrics",         has_ped, "pediatrics",         "specialties"),
        ("has_icu",                has_icu, "criticalCareMedicine","specialties"),
        ("has_radiology",          has_rad, "radiology",          "specialties"),
        ("has_infectious_disease", has_inf, "infectiousDiseases", "specialties"),
        ("has_mental_health",      has_mh,  "psychiatry",         "specialties"),
    ]
    for flag_name, fired, snip_val, src_col in FLAG_SOURCES:
        if not fired:
            continue
        # Find supporting evidence snippet
        snippet = snip_val
        # Try to find the actual item in the arrays
        for arr in [spec_arr, cap_arr, proc_arr, equip_arr]:
            if arr:
                for item in arr:
                    if item and snip_val.lower() in str(item).lower():
                        snippet = str(item)[:100]
                        break
        key = (flag_name, snippet[:50])
        if key not in seen_snippets:
            seen_snippets.add(key)
            out.append({
                "field":             flag_name,
                "snippet":           snippet[:300],
                "source_column":     src_col,
                "extraction_method": "specialty_flag_trigger",
                "confidence":        float(0.92),
                "step_id":           f"gold_v6_flag_{flag_name}",
            })

    # Ensure at least a description fallback if nothing else
    if not out:
        out.append({
            "field":             "description",
            "snippet":           "",
            "source_column":     "description",
            "extraction_method": "no_evidence",
            "confidence":        float(0.0),
            "step_id":           "gold_v6_no_evidence",
        })

    return out


upgrade_citations_udf = F.udf(_upgrade_citations, GOLD_CITATION_SCHEMA)

gold = gold.withColumn(
    "citations",
    upgrade_citations_udf(
        F.col("citations"),
        F.col("procedure_parsed"),
        F.col("equipment_parsed"),
        F.col("capability_parsed"),
        F.col("specialties_parsed"),
        F.col("has_emergency_medicine"),
        F.col("has_obstetrics"),
        F.col("has_surgery"),
        F.col("has_pediatrics"),
        F.col("has_icu"),
        F.col("has_radiology"),
        F.col("has_infectious_disease"),
        F.col("has_mental_health"),
    )
)

print("Citations upgraded with step_id. Sample:")
gold.select("name", F.size("citations").alias("citation_count")).show(8, truncate=60)

# COMMAND ----------
# MAGIC %md ## 10. Preliminary Facility-Level Medical Desert Score
# MAGIC
# MAGIC `medical_desert_score = 1.0 - data_completeness_score`
# MAGIC (proxy; authoritative regional score written back by Notebook 07)

# COMMAND ----------

gold = gold \
    .withColumn("medical_desert_score",
                (F.lit(1.0) - F.col("data_completeness_score")).cast(FloatType())) \
    .withColumn("desert_label",
                F.when(F.col("medical_desert_score") >= 0.8, F.lit("Severe Desert"))
                 .when(F.col("medical_desert_score") >= 0.6, F.lit("Moderate Desert"))
                 .when(F.col("medical_desert_score") >= 0.4, F.lit("Marginal"))
                 .when(F.col("medical_desert_score") >= 0.2, F.lit("Adequate"))
                 .otherwise(F.lit("Well-Served")))

print("Preliminary desert scores:")
gold.groupBy("desert_label").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 11. Update extraction_version

# COMMAND ----------

gold = gold.withColumn("extraction_version", F.lit(cfg.EXTRACTION_VER))

# COMMAND ----------
# MAGIC %md ## 12. Final Column Selection — Exact Gold Schema (113 columns)

# COMMAND ----------

# Exact column list in exact schema order from gold_facilities_enriched DDL
GOLD_COLUMNS = [
    # Region first
    "region_normalised",
    # Identity
    "unique_id", "source_url", "name", "pk_unique_id", "mongo_db",
    # Raw JSON source fields (STRING)
    "specialties", "procedure", "equipment", "capability",
    "organization_type", "content_table_id",
    # phone_numbers is ARRAY<STRING> in gold (converted above)
    "phone_numbers",
    "email", "websites", "officialWebsite",
    "yearestablished", "acceptsvolunteers",
    "facebooklink", "twitterlink", "linkedinlink", "instagramlink", "logo",
    # Address
    "address_line1", "address_line2", "address_line3",
    "address_city", "address_stateOrRegion", "address_zipOrPostcode",
    "address_country", "address_countryCode",
    "countries",
    # Organisation text
    "missionstatement", "missionstatementlink", "organizationdescription",
    # Type IDs (STRING)
    "facilityTypeId", "operatorTypeId", "affiliationtypeids",
    # Clinical text
    "description", "area",
    # Numeric (INT in gold schema)
    "numberDoctors",  # INT
    "capacity",       # STRING (original)
    # Bronze metadata
    "ingested_at", "source_file", "dataset_version", "country_scope", "row_hash",
    # Parsed arrays
    "specialties_parsed", "procedure_parsed", "equipment_parsed", "capability_parsed",
    "phone_numbers_parsed", "affiliationtypeids_parsed", "countries_parsed", "websites_parsed",
    # Derived single-value fields
    "official_phone", "area_int", "year_established_int",
    "accepts_volunteers_bool", "pk_unique_id_int",
    # Classification
    "facility_type_clean", "facility_type_confidence",
    "organization_type_clean", "city_clean",
    "capacity_int", "number_doctors_int",
    # Region (derived)
    "region_source", "region_confidence",
    # Geocoding
    "latitude", "longitude", "geo_source", "geo_confidence",
    # Content flags
    "has_procedures", "has_equipment", "has_capabilities",
    "has_specialties", "has_description", "has_contact",
    # Counts
    "procedure_count", "equipment_count", "capability_count", "specialty_count",
    # RAG
    "doc_text", "doc_text_length", "is_rag_ready",
    # Provenance (ARRAY<STRUCT> with step_id)
    "citations",
    # Quality
    "row_quality_flags", "quality_flag_count", "data_completeness_score",
    "extraction_version",
    # Gold-only specialty flags
    "has_emergency_medicine", "has_obstetrics", "has_surgery", "has_pediatrics",
    "has_icu", "has_radiology", "has_infectious_disease", "has_mental_health",
    # Operator / classification booleans
    "is_public", "is_private", "is_hospital", "is_clinic", "is_ngo",
    # Anomaly flags
    "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
    "stat_anomaly_clinic_claims_icu",    "stat_anomaly_ghost_facility",
    "stat_anomaly_specialty_mismatch",   "stat_anomaly_procedure_breadth",
    "total_stat_anomalies",
    # NGO filter
    "ngo_serves_ghana",
    # Citation ID
    "citation_row_id",
    # Desert score (facility-level proxy)
    "medical_desert_score", "desert_label",
]

# Validate presence
missing_cols = [c for c in GOLD_COLUMNS if c not in gold.columns]
if missing_cols:
    raise ValueError(f"MISSING columns before write: {missing_cols}")

extra_cols = [c for c in gold.columns if c not in GOLD_COLUMNS]
print(f"Extra columns to drop: {extra_cols}")

gold_final = gold.select(*GOLD_COLUMNS)

actual_cnt = len(gold_final.columns)
expected_cnt = len(GOLD_COLUMNS)

print(f"\nColumn count: {actual_cnt} (expected {expected_cnt})")

assert actual_cnt == expected_cnt, \
    f"Column mismatch: {actual_cnt} vs {expected_cnt}"

# COMMAND ----------
# MAGIC %md ## 13. Write gold_facilities_enriched

# COMMAND ----------

(
    gold_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact",   "true")
    .saveAsTable(cfg.GOLD_FACILITIES)
)

_readback = spark.table(cfg.GOLD_FACILITIES)
_rc = _readback.count()
_cc = len(_readback.columns)
print(f"✅  {cfg.GOLD_FACILITIES}")
print(f"    Rows    : {_rc:,}")
print(f"    Columns : {_cc}")
assert _rc == gold.count(), "Row count mismatch after write!"
assert _cc == len(GOLD_COLUMNS), \
    f"Column count mismatch after write: {_cc} vs {len(GOLD_COLUMNS)}"
read_cols = set(_readback.columns)
expected_cols = set(GOLD_COLUMNS)

missing = expected_cols - read_cols
extra   = read_cols - expected_cols

print("Missing after write:", missing)
print("Extra after write:", extra)

assert not missing, f"Missing columns after write: {missing}"
assert not extra,   f"Unexpected columns after write: {extra}"
# Register comment
spark.sql(f"""
    COMMENT ON TABLE {cfg.GOLD_FACILITIES}
    IS 'Gold facilities: specialty-flagged, anomaly-scored, citation-enriched — {cfg.EXTRACTION_VER}'
""")

# COMMAND ----------
# MAGIC %md ## 14. Build gold_regional_summary (37 columns)

# COMMAND ----------

gf = spark.table(cfg.GOLD_FACILITIES)

CRITICAL_SPECS_5 = [
    "emergencyMedicine", "generalSurgery",
    "gynecologyAndObstetrics", "pediatrics", "internalMedicine",
]

# ── Core aggregation ───────────────────────────────────────────────────────
regional = gf.groupBy("region_normalised").agg(
    # Facility counts
    F.count("unique_id")                           .alias("total_facilities"),
    F.sum(F.when(F.col("is_hospital") | F.col("is_clinic"), 1).otherwise(0))
                                                   .alias("clinical_facility_count"),
    F.sum(F.col("is_hospital").cast(IntegerType())).alias("hospital_count"),
    F.sum(F.when(
        F.col("is_hospital") & (
            F.col("has_procedures") | F.col("has_equipment") | F.col("has_capabilities")
        ), 1).otherwise(0))                        .alias("clinical_hospital_count"),
    F.sum(F.col("is_clinic").cast(IntegerType()))  .alias("clinic_count"),
    F.sum(F.col("is_public").cast(IntegerType()))  .alias("public_facilities"),
    F.sum(F.col("is_private").cast(IntegerType())) .alias("private_facilities"),
    F.sum(F.col("is_ngo").cast(IntegerType()))     .alias("ngo_count"),

    # Medical resources
    F.avg("number_doctors_int")                    .alias("avg_doctors"),
    F.sum(F.coalesce("number_doctors_int", F.lit(0))).alias("total_doctors"),
    F.avg("capacity_int")                          .alias("avg_bed_capacity"),
    F.sum(F.coalesce("capacity_int", F.lit(0)))    .alias("total_beds"),

    # Completeness
    F.avg("data_completeness_score")               .alias("avg_completeness"),

    # Specialty facility counts
    F.sum(F.col("has_emergency_medicine").cast(IntegerType())).alias("emergency_medicine_facilities"),
    F.sum(F.col("has_obstetrics")        .cast(IntegerType())).alias("obstetrics_facilities"),
    F.sum(F.col("has_surgery")           .cast(IntegerType())).alias("surgery_facilities"),
    F.sum(F.col("has_pediatrics")        .cast(IntegerType())).alias("pediatrics_facilities"),
    F.sum(F.col("has_icu")               .cast(IntegerType())).alias("icu_facilities"),
    F.sum(F.col("has_infectious_disease").cast(IntegerType())).alias("infectious_disease_facilities"),
    F.sum(F.col("has_radiology")         .cast(IntegerType())).alias("radiology_facilities"),
    F.sum(F.col("has_mental_health")     .cast(IntegerType())).alias("mental_health_facilities"),

    # Infrastructure counts
    F.sum(F.col("has_procedures") .cast(IntegerType())).alias("facilities_with_procedures"),
    F.sum(F.col("has_equipment")  .cast(IntegerType())).alias("facilities_with_equipment"),
    F.sum(F.col("has_capabilities").cast(IntegerType())).alias("facilities_with_capabilities"),
    F.sum(F.when(F.col("accepts_volunteers_bool") == True, 1).otherwise(0))
                                                       .alias("volunteer_facilities"),

    # Geospatial (first valid lat/lon in region)
    F.first(F.col("latitude"),  ignorenulls=True)  .alias("region_centroid_lat"),
    F.first(F.col("longitude"), ignorenulls=True)  .alias("region_centroid_lon"),

    # Anomaly counts
    F.sum("total_stat_anomalies")                  .alias("total_region_anomalies"),
    F.sum(F.col("stat_anomaly_procedure_breadth").cast(IntegerType()))
                                                   .alias("procedure_breadth_anomalies"),
    # RAG readiness
    F.sum(F.col("is_rag_ready").cast(IntegerType())).alias("rag_ready_count"),

    # All specialties in region (collect as set, flatten)
    F.collect_list("specialties_parsed")           .alias("_all_spec_arrays"),
)

# ── Flatten and deduplicate all_specialties ────────────────────────────────
_CANONICAL_SPEC_SET = frozenset([
    "internalMedicine", "familyMedicine", "pediatrics", "cardiology",
    "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics",
    "orthopedicSurgery", "dentistry", "ophthalmology", "radiology",
    "pathology", "infectiousDiseases", "nephrology", "criticalCareMedicine",
    "cardiacSurgery", "plasticSurgery", "neurology", "psychiatry", "anesthesia",
    "dermatology", "urology", "gastroenterology", "pulmonology",
    "endocrinologyAndDiabetesAndMetabolism", "neonatologyPerinatalMedicine",
    "medicalOncology", "physicalMedicineAndRehabilitation", "otolaryngology",
    "geriatricsInternalMedicine", "hospiceAndPalliativeInternalMedicine",
    "publicHealth", "globalHealthAndInternationalHealth",
    "maternalFetalMedicineOrPerinatology", "socialAndBehavioralSciences",
    "obstetricsAndMaternityCare", "reproductiveEndocrinologyAndInfertility",
    "clinicalPathology", "anatomicPathology", "diagnosticRadiology",
    "interventionalRadiology", "neurosurgery", "pediatricSurgery", "vascularSurgery",
])


@F.udf(ArrayType(StringType()))
def flatten_filter_specs(all_arrays):
    """Flatten nested arrays, filter to canonical taxonomy, sort, deduplicate."""
    if not all_arrays:
        return []
    seen, out = set(), []
    for sub_arr in all_arrays:
        if not sub_arr:
            continue
        for s in sub_arr:
            if s and s in _CANONICAL_SPEC_SET and s not in seen:
                seen.add(s)
                out.append(s)
    return sorted(out)


regional = regional \
    .withColumn("all_specialties", flatten_filter_specs(F.col("_all_spec_arrays"))) \
    .drop("_all_spec_arrays")

# ── Missing critical specialties ───────────────────────────────────────────
@F.udf(ArrayType(StringType()))
def missing_critical_specialties_udf(all_specs):
    """Return which of the 5 critical specialties are missing in this region."""
    present = set(all_specs or [])
    return [s for s in CRITICAL_SPECS_5 if s not in present]


regional = regional \
    .withColumn("missing_critical_specialties",
                missing_critical_specialties_udf(F.col("all_specialties"))) \
    .withColumn("critical_specialty_gap_count",
                F.size(F.col("missing_critical_specialties")).cast(IntegerType()))

# ── Recommended actions ────────────────────────────────────────────────────
@F.udf(ArrayType(StringType()))
def recommended_actions_udf(
    missing_specs, avg_doctors, total_beds,
    icu_fac, hospital_count, mental_fac, total_fac,
):
    actions = []
    missing = list(missing_specs or [])

    if "emergencyMedicine" in missing:
        actions.append("URGENT: Deploy emergency medicine capacity")
    if "generalSurgery" in missing and (hospital_count or 0) > 0:
        actions.append("URGENT: No surgical capacity — patients cannot receive operative care")
    if "gynecologyAndObstetrics" in missing:
        actions.append("URGENT: No obstetrics — maternal mortality risk elevated")
    if "pediatrics" in missing:
        actions.append("Deploy paediatric care — children's health services absent")
    if "internalMedicine" in missing:
        actions.append("Recruit internal medicine specialists")

    if (avg_doctors or 0.0) < 1.0:
        actions.append("URGENT: Physician recruitment required — avg doctors < 1")
    if (total_beds or 0) < 10:
        actions.append("Deploy mobile inpatient capacity — fewer than 10 beds in region")
    if (icu_fac or 0) == 0 and (hospital_count or 0) > 0:
        actions.append("No ICU capacity — critical gap for complex cases")
    if (mental_fac or 0) == 0:
        actions.append("Mental health services absent")
    if (total_fac or 0) < 3:
        actions.append("Region severely underserved — prioritize facility establishment")

    return actions if actions else ["Monitor — no critical gaps identified"]


regional = regional.withColumn(
    "recommended_actions",
    recommended_actions_udf(
        F.col("missing_critical_specialties"),
        F.col("avg_doctors"),
        F.col("total_beds"),
        F.col("icu_facilities"),
        F.col("hospital_count"),
        F.col("mental_health_facilities"),
        F.col("total_facilities"),
    )
)

# ── Regional medical desert score & label (placeholder → Notebook 07 overwrites) ──
regional = regional \
    .withColumn("medical_desert_score", F.lit(None).cast(FloatType())) \
    .withColumn("desert_label",         F.lit("Pending").cast(StringType()))

# ── Validate schema (37 columns) ──────────────────────────────────────────
REGIONAL_COLUMNS = [
    "region_normalised",
    "total_facilities", "clinical_facility_count", "hospital_count",
    "clinical_hospital_count", "clinic_count",
    "public_facilities", "private_facilities", "ngo_count",
    "avg_doctors", "total_doctors", "avg_bed_capacity", "total_beds",
    "avg_completeness",
    "emergency_medicine_facilities", "obstetrics_facilities",
    "surgery_facilities", "pediatrics_facilities", "icu_facilities",
    "infectious_disease_facilities", "radiology_facilities",
    "mental_health_facilities",
    "facilities_with_procedures", "facilities_with_equipment",
    "facilities_with_capabilities", "volunteer_facilities",
    "region_centroid_lat", "region_centroid_lon",
    "total_region_anomalies", "procedure_breadth_anomalies", "rag_ready_count",
    "all_specialties", "missing_critical_specialties",
    "critical_specialty_gap_count", "recommended_actions",
    "medical_desert_score", "desert_label",
]

_miss_reg = [c for c in REGIONAL_COLUMNS if c not in regional.columns]
if _miss_reg:
    raise ValueError(f"Missing regional columns: {_miss_reg}")

regional_final = regional.select(*REGIONAL_COLUMNS)
print(f"Regional column count: {len(regional_final.columns)}  (expected 37)")
assert len(regional_final.columns) == 37

# COMMAND ----------
# MAGIC %md ## 15. Write gold_regional_summary

# COMMAND ----------

(
    regional_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.GOLD_REGIONAL)
)

_reg_cnt = spark.table(cfg.GOLD_REGIONAL).count()
print(f"✅  {cfg.GOLD_REGIONAL}")
print(f"    Rows    : {_reg_cnt}  (expected 16–17)")
print(f"    Columns : {len(regional_final.columns)}")

spark.sql(f"""
    COMMENT ON TABLE {cfg.GOLD_REGIONAL}
    IS 'Regional summary with specialty coverage, desert scores (placeholder), anomaly counts'
""")

# COMMAND ----------
# MAGIC %md ## 16. Gold Quality Report

# COMMAND ----------

gf  = spark.table(cfg.GOLD_FACILITIES)
gr  = spark.table(cfg.GOLD_REGIONAL)
tot = gf.count()

print(f"{'='*68}")
print(f"GOLD FACILITIES — Quality Report  ({tot:,} rows, {len(gf.columns)} cols)")
print(f"{'='*68}")

# Key metrics
checks = [
    ("is_hospital",               gf.filter(F.col("is_hospital")).count()),
    ("is_clinic",                 gf.filter(F.col("is_clinic")).count()),
    ("is_ngo",                    gf.filter(F.col("is_ngo")).count()),
    ("has_emergency_medicine",    gf.filter(F.col("has_emergency_medicine")).count()),
    ("has_obstetrics",            gf.filter(F.col("has_obstetrics")).count()),
    ("has_surgery",               gf.filter(F.col("has_surgery")).count()),
    ("has_icu",                   gf.filter(F.col("has_icu")).count()),
    ("has_radiology",             gf.filter(F.col("has_radiology")).count()),
    ("has_infectious_disease",    gf.filter(F.col("has_infectious_disease")).count()),
    ("has_mental_health",         gf.filter(F.col("has_mental_health")).count()),
    ("is_rag_ready",              gf.filter(F.col("is_rag_ready")).count()),
    ("stat_anomaly_cap_inflation",gf.filter(F.col("stat_anomaly_capability_inflation")).count()),
    ("stat_anomaly_hosp_no_docs", gf.filter(F.col("stat_anomaly_hospital_no_doctors")).count()),
    ("stat_anomaly_ghost",        gf.filter(F.col("stat_anomaly_ghost_facility")).count()),
    ("stat_anomaly_clinic_icu",   gf.filter(F.col("stat_anomaly_clinic_claims_icu")).count()),
    ("stat_anomaly_spec_mismatch",gf.filter(F.col("stat_anomaly_specialty_mismatch")).count()),
    ("stat_anomaly_proc_breadth", gf.filter(F.col("stat_anomaly_procedure_breadth")).count()),
    ("total_stat_anomalies > 0",  gf.filter(F.col("total_stat_anomalies") > 0).count()),
    ("ngo_serves_ghana",          gf.filter(F.col("ngo_serves_ghana")).count()),
    ("citations non-empty",       gf.filter(F.size("citations") > 0).count()),
]
for label, cnt in checks:
    pct = cnt / tot * 100
    print(f"  {label:<40}  {cnt:>5,}  ({pct:5.1f}%)")

print()
print("Desert label distribution:")
gf.groupBy("desert_label").count().orderBy(F.desc("count")).show()

print(f"{'='*68}")
print(f"GOLD REGIONAL SUMMARY  ({gr.count()} regions)")
print(f"{'='*68}")
display(
    gr.select(
        "region_normalised", "total_facilities", "hospital_count", "ngo_count",
        "total_doctors", "total_beds",
        "emergency_medicine_facilities", "icu_facilities",
        "critical_specialty_gap_count", "missing_critical_specialties",
        "avg_completeness", "total_region_anomalies",
    ).orderBy(F.desc("total_facilities"))
)

# COMMAND ----------
# MAGIC %md ## 17. MLflow Tracing

# COMMAND ----------

try:
    import mlflow

    mlflow.set_experiment("/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon")

    with mlflow.start_run(run_name="03_gold_build_v6") as run:
        # Core row counts
        mlflow.log_metric("total_facilities",         tot)
        mlflow.log_metric("hospital_count",           gf.filter(F.col("is_hospital")).count())
        mlflow.log_metric("clinic_count",             gf.filter(F.col("is_clinic")).count())
        mlflow.log_metric("ngo_count",                gf.filter(F.col("is_ngo")).count())
        mlflow.log_metric("rag_ready_count",          gf.filter(F.col("is_rag_ready")).count())
        mlflow.log_metric("unknown_region_pct",       round(unk_ct / total * 100, 2))

        # Specialty flags
        for _f in ["has_emergency_medicine","has_obstetrics","has_surgery","has_pediatrics",
                   "has_icu","has_radiology","has_infectious_disease","has_mental_health"]:
            mlflow.log_metric(_f, gf.filter(F.col(_f)).count())

        # Anomaly flags
        for _f in ["stat_anomaly_capability_inflation","stat_anomaly_hospital_no_doctors",
                   "stat_anomaly_clinic_claims_icu","stat_anomaly_ghost_facility",
                   "stat_anomaly_specialty_mismatch","stat_anomaly_procedure_breadth"]:
            mlflow.log_metric(_f, gf.filter(F.col(_f)).count())

        mlflow.log_metric("total_anomaly_flag_sum",
                          gf.agg(F.sum("total_stat_anomalies")).collect()[0][0] or 0)

        # Regional summary
        mlflow.log_metric("regions_total",       gr.count())
        mlflow.log_metric("regions_canonical",
                          gr.filter(F.col("region_normalised") != "Unknown").count())

        # Params
        mlflow.log_param("gold_facilities_table", cfg.GOLD_FACILITIES)
        mlflow.log_param("gold_regional_table",   cfg.GOLD_REGIONAL)
        mlflow.log_param("extraction_version",    cfg.EXTRACTION_VER)
        mlflow.log_param("gold_col_count",        str(len(gf.columns)))
        mlflow.log_param("regional_col_count",    str(len(gr.columns)))

        print(f"✅  MLflow run: {run.info.run_id}")

except Exception as e:
    print(f"MLflow skipped: {e}")
# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT
# MAGIC   unique_id, name, region_normalised, facilityTypeId, operatorTypeId,
# MAGIC   officialWebsite, address_stateOrRegion, address_countryCode,
# MAGIC   numberDoctors, capacity_int, medical_desert_score, desert_label,
# MAGIC   has_emergency_medicine, has_surgery, has_icu,
# MAGIC   total_stat_anomalies, is_rag_ready
# MAGIC FROM virtue_foundation.ghana.gold_facilities_enriched
# MAGIC ORDER BY medical_desert_score DESC
# MAGIC LIMIT 50

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.gold_regional_summary
# MAGIC ORDER BY medical_desert_score DESC

#######################################
# OUTPUT OF THE NOTEBOOK
######################################

'''
Run : 2026-05-02T02:40:31.736469+00:00
Spark: 4.1.0
Silver rows    : 987
Silver columns : 90
Region distribution after second-pass consolidation:
+-----------------+-----+
|region_normalised|count|
+-----------------+-----+
|    Greater Accra|  447|
|          Ashanti|  170|
|          Western|   82|
|          Unknown|   62|
|          Central|   42|
|            Volta|   36|
|         Northern|   31|
|      Brong-Ahafo|   28|
|          Eastern|   27|
|        Bono East|   16|
|    Western North|   10|
|       Upper West|   10|
|              Oti|    8|
|            Ahafo|    7|
|       Upper East|    5|
|         Savannah|    4|
|       North East|    2|
+-----------------+-----+

Unknown region: 62 / 987  (6.3%)  target < 15%
Second-pass cleaning complete. Array count distributions:
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)
  procedure_count       avg=1.27  max=26  non-zero=279
  equipment_count       avg=0.27  max=13  non-zero=92
  capability_count      avg=3.12  max=20  non-zero=804
  specialty_count       avg=1.89  max=23  non-zero=841
phone_numbers column converted to ARRAY<STRING>.
+------------------------------------------------------+------------------------------+------------------------------+
|                                                  name|                 phone_numbers|          phone_numbers_parsed|
+------------------------------------------------------+------------------------------+------------------------------+
|109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana|[+233249354576, +233203928883]|[+233249354576, +233203928883]|
|                                 1st Foundation Clinic|                            []|                            []|
|                                 1st Foundation Clinic|                            []|                            []|
|                                 1st Foundation Clinic|                            []|                            []|
|                                 1st Foundation Clinic|                            []|                            []|
+------------------------------------------------------+------------------------------+------------------------------+
only showing top 5 rows
Specialty flags computed. Counts:
  has_emergency_medicine            68 / 987  (6.9%)
  has_obstetrics                   187 / 987  (18.9%)
  has_surgery                      109 / 987  (11.0%)
  has_pediatrics                   113 / 987  (11.4%)
  has_icu                           13 / 987  (1.3%)
  has_radiology                     64 / 987  (6.5%)
  has_infectious_disease            35 / 987  (3.5%)
  has_mental_health                 38 / 987  (3.9%)
Operator / classification flags:
  is_hospital          498
  is_clinic            391
  is_ngo                67
  is_public             63
  is_private           143
ngo_serves_ghana: 67
Anomaly flags computed:
  stat_anomaly_capability_inflation             12 (1.2%)
  stat_anomaly_hospital_no_doctors             494 (50.1%)
  stat_anomaly_clinic_claims_icu                 1 (0.1%)
  stat_anomaly_ghost_facility                    0 (0.0%)
  stat_anomaly_specialty_mismatch                0 (0.0%)
  stat_anomaly_procedure_breadth                16 (1.6%)
  Rows with at least 1 anomaly: 509
Citations upgraded with step_id. Sample:
+------------------------------------------------------+--------------+
|                                                  name|citation_count|
+------------------------------------------------------+--------------+
|109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana|             7|
|                                 1st Foundation Clinic|             1|
|                                 1st Foundation Clinic|             1|
|                                 1st Foundation Clinic|             1|
|                                 1st Foundation Clinic|             1|
|                                 2BN Military Hospital|             3|
|                                  37 Military Hospital|             1|
|                                  37 Military Hospital|             1|
+------------------------------------------------------+--------------+
only showing top 8 rows
Preliminary desert scores:
+---------------+-----+
|   desert_label|count|
+---------------+-----+
|    Well-Served|  531|
|       Adequate|  428|
|       Marginal|   24|
|Moderate Desert|    4|
+---------------+-----+

Extra columns to drop: []

Column count: 114 (expected 114)
✅  virtue_foundation.ghana.gold_facilities_enriched
    Rows    : 987
    Columns : 114
Missing after write: set()
Extra after write: set()
Regional column count: 37  (expected 37)
✅  virtue_foundation.ghana.gold_regional_summary
    Rows    : 17  (expected 16–17)
    Columns : 37
====================================================================
GOLD FACILITIES — Quality Report  (987 rows, 114 cols)
====================================================================
  is_hospital                                 498  ( 50.5%)
  is_clinic                                   391  ( 39.6%)
  is_ngo                                       67  (  6.8%)
  has_emergency_medicine                       68  (  6.9%)
  has_obstetrics                              187  ( 18.9%)
  has_surgery                                 109  ( 11.0%)
  has_icu                                      13  (  1.3%)
  has_radiology                                64  (  6.5%)
  has_infectious_disease                       35  (  3.5%)
  has_mental_health                            38  (  3.9%)
  is_rag_ready                                980  ( 99.3%)
  stat_anomaly_cap_inflation                   12  (  1.2%)
  stat_anomaly_hosp_no_docs                   494  ( 50.1%)
  stat_anomaly_ghost                            0  (  0.0%)
  stat_anomaly_clinic_icu                       1  (  0.1%)
  stat_anomaly_spec_mismatch                    0  (  0.0%)
  stat_anomaly_proc_breadth                    16  (  1.6%)
  total_stat_anomalies > 0                    509  ( 51.6%)
  ngo_serves_ghana                             67  (  6.8%)
  citations non-empty                         987  (100.0%)

Desert label distribution:
+---------------+-----+
|   desert_label|count|
+---------------+-----+
|    Well-Served|  531|
|       Adequate|  428|
|       Marginal|   24|
|Moderate Desert|    4|
+---------------+-----+

====================================================================
GOLD REGIONAL SUMMARY  (17 regions)
====================================================================
'''

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.gold_facilities_enriched

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_facilities_enriched 

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_facilities_enriched